<a href="https://colab.research.google.com/github/fc63/gender-classification/blob/main/tokenized/tokenized.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datasets transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 104.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import warnings
import pandas as pd
import numpy as np
import os
import re
import pickle
import gc
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import Dataset as HFDataset
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from google.colab import drive
from transformers import EarlyStoppingCallback

drive.mount('/content/drive')

with open('/content/drive/MyDrive/datasets/europarl_normalized.pkl', 'rb') as f:
    df = pickle.load(f)

Mounted at /content/drive


In [3]:
with open('/content/drive/MyDrive/datasets/europarl_normalized.pkl', 'wb') as f:
    pickle.dump(df, f)

In [4]:
print(df['gender'].value_counts())

min_count = df['gender'].value_counts().min()

df = (
    df.groupby('gender', group_keys=False)
    .apply(lambda x: x.sample(n=min_count, random_state=63))
    .reset_index(drop=True)
)

print(df['gender'].value_counts())

gender
male      740894
female    360361
Name: count, dtype: int64
gender
female    360361
male      360361
Name: count, dtype: int64


<ipython-input-4-e34e100695cf>:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min_count, random_state=63))


In [5]:
# encode labels
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['gender'])

df

,text,gender,label
0,"Cooperation is not only in our interest, but i...",female,0
1,"During the committee's first meeting, we enter...",female,0
2,"Towards the end of the debate on the matter, a...",female,0
3,"After all, something is obviously being set in...",female,0
4,"Today, the new Member States in particular req...",female,0
...,...,...,...
720717,"Their function, as I understand it, is to diss...",male,1
720718,That is my first point.,male,1
720719,What we need to do tomorrow is to accept the c...,male,1
720720,in writing. I welcome this report which:,male,1


In [6]:
print(label_encoder.classes_)
print(df['label'].value_counts())

['female' 'male']
label
0    360361
1    360361
Name: count, dtype: int64


In [7]:
# dataset tokenized and saved as pickle

tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-large", use_fast=False)
hf_dataset = HFDataset.from_pandas(df[['text', 'label']])

def tokenize_function(example):
    return tokenizer(
        example['text'],
        truncation=True,
        padding='max_length',
        max_length=256
    )

tokenized_dataset = hf_dataset.map(tokenize_function)

tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.21, seed=63)
train_dataset = tokenized_dataset['train']
test_dataset = tokenized_dataset['test']

os.makedirs('/content/drive/MyDrive/datasets', exist_ok=True)

with open('/content/drive/MyDrive/datasets/tokenized_train.pkl', 'wb') as f:
    pickle.dump(train_dataset, f)

with open('/content/drive/MyDrive/datasets/tokenized_test.pkl', 'wb') as f:
    pickle.dump(test_dataset, f)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Map:   0%|          | 0/720722 [00:00<?, ? examples/s]